# Hands-on Activity 8.1: Aggregating Data with Pandas
### 8.1.1 Intended Learning Outcomes
After this activity, the student should be able to:
* Demonstrate querying and merging of dataframes
* Perform advanced calculations on dataframes
* Aggregate dataframes with pandas and numpy
* Work with time series data
### 8.1.2 Resources
* Computing Environment using Python 3.x
* Attached Datasets (under Instructional Materials)
### 8.1.3 Procedures
The procedures can be found in the canvas module. Check the following under topics:
* 8.1 Weather Data Collection
* 8.2 Querying and Merging
* 8.3 Dataframe Operations
* 8.4 Aggregations
* 8.5 Time Series
### 8.1.4 Data Analysis
Provide some comments here about the results of the procedures.
### 8.1.5 Supplementary Activity
Using the CSV files provided and what we have learned so far in this module complete the following exercises:
1. With the earthquakes.csv file, select all the earthquakes in Japan with a magType of mb and a magnitude of 4.9 or greater.
2. Create bins for each full number of magnitude (for example, the first bin is 0-1, the second is 1-2, and so on) with a magType of ml and count how many are in each bin.
3. Using the faang.csv file, group by the ticker and resample to monthly frequency. Make the following aggregations:
    * Mean of the opening price
    * Maximum of the high price
    * Minimum of the low price
    * Mean of the closing price
    * Sum of the volume traded
4. Build a crosstab with the earthquake data between the tsunami column and the magType column. Rather than showing the frequency count, show the maximum
magnitude that was observed for each combination. Put the magType along the columns.
5. Calculate the rolling 60-day aggregations of OHLC data by ticker for the FAANG data. Use the same aggregations as exercise no. 3.
6. Create a pivot table of the FAANG data that compares the stocks. Put the ticker in the rows and show the averages of the OHLC and volume traded data.
7. Calculate the Z-scores for each numeric column of Netflix's data (ticker is NFLX) using apply().
8. Add event descriptions:
    * Create a dataframe with the following three columns: ticker, date, and event. The columns should have the following values:
        * ticker: 'FB'
        * date: ['2018-07-25', '2018-03-19', '2018-03-20']
        * event: ['Disappointing user growth announced after close.', 'Cambridge Analytica story', 'FTC investigation']
    * Set the index to ['date', 'ticker']
    * Merge this data with the FAANG data using an outer join
9. Use the transform() method on the FAANG data to represent all the values in terms of the first date in the data. To do so, divide all the values for each ticker by the values
for the first date in the data for that ticker. This is referred to as an index, and the data for the first date is the base (https://ec.europa.eu/eurostat/statistics-explained/index.php/ Beginners:Statisticalconcept-Indexandbaseyear). When data is in this format, we can easily see growth over time. Hint: transform() can take a function name.

In [29]:
# 1
import pandas as pd

earthquakes = pd.read_csv('/content/earthquakes.csv')
japan_earthqueake = earthquakes[(earthquakes['parsed_place'] == 'Japan') & (earthquakes['magType']== 'mb') & (earthquakes['mag'] >= 4.9)]
japan_earthqueake

,mag,magType,time,place,tsunami,parsed_place
1563,4.9,mb,1538977532250,"293km ESE of Iwo Jima, Japan",0,Japan
2576,5.4,mb,1538697528010,"37km E of Tomakomai, Japan",0,Japan
3072,4.9,mb,1538579732490,"15km ENE of Hasaki, Japan",0,Japan
3632,4.9,mb,1538450871260,"53km ESE of Hitachi, Japan",0,Japan


In [30]:
# 2
ml_quakes = earthquakes[earthquakes['magType'] == 'ml'].copy()

bins = range(0, int(ml_quakes['mag'].max()) + 2)

ml_quakes['mag_bin'] = pd.cut(
    ml_quakes['mag'],
    bins=bins,
    right=False
)

bin_counts = ml_quakes['mag_bin'].value_counts().sort_index()

bin_counts

,count
mag_bin,
"[0, 1)",2072
"[1, 2)",3126
"[2, 3)",985
"[3, 4)",153
"[4, 5)",6
"[5, 6)",2


In [31]:
#3
faang = pd.read_csv('faang.csv', parse_dates=['date'])
faang.set_index('date', inplace=True)

monthly = (
    faang
    .groupby('ticker')
    .resample('M')
    .agg({
        'open': 'mean',
        'high': 'max',
        'low': 'min',
        'close': 'mean',
        'volume': 'sum'
    })
)

monthly

/tmp/ipython-input-207/3155902090.py:8: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  .resample('M')


open       high        low        close     volume
ticker date                                                                 
AAPL   2018-01-31   170.714690   176.6782   161.5708   170.699271  659679440
       2018-02-28   164.562753   177.9059   147.9865   164.921884  927894473
       2018-03-31   172.421381   180.7477   162.4660   171.878919  713727447
       2018-04-30   167.332895   176.2526   158.2207   167.286924  666360147
       2018-05-31   182.635582   187.9311   162.7911   183.207418  620976206
       2018-06-30   186.605843   192.0247   178.7056   186.508652  527624365
       2018-07-31   188.065786   193.7650   181.3655   188.179724  393843881
       2018-08-31   210.460287   227.1001   195.0999   211.477743  700318837
       2018-09-30   220.611742   227.8939   213.6351   220.356353  678972040
       2018-10-31   219.489426   231.6645   204.4963   219.137822  789748068
       2018-11-30   190.828681   220.6405   169.5328   190.246652  961321947
       2018-12-31   164.537405   184.1501   145.9639   163.564732  898917007
AMZN   2018-01-31  1301.377143  1472.5800  1170.5100  1309.010952   96371290
       2018-02-28  1447.112632  1528.7000  1265.9300  1442.363158  137784020
       2018-03-31  1542.160476  1617.5400  1365.2000  1540.367619  130400151
       2018-04-30  1475.841905  1638.1000  1352.8800  1468.220476  129945743
       2018-05-31  1590.474545  1635.0000  1546.0200  1594.903636   71615299
       2018-06-30  1699.088571  1763.1000  1635.0900  1698.823810   85941510
       2018-07-31  1786.305714  1880.0500  1678.0600  1784.649048   97629820
       2018-08-31  1891.957826  2025.5700  1776.0200  1897.851304   96575676
       2018-09-30  1969.239474  2050.5000  1865.0000  1966.077895   94445693
       2018-10-31  1799.630870  2033.1900  1476.3600  1782.058261  183228552
       2018-11-30  1622.323810  1784.0000  1420.0000  1625.483810  139290208
       2018-12-31  1572.922105  1778.3400  1307.0000  1559.443158  154812304
FB     2018-01-31   184.364762   190.6600   175.8000   184.962857  495655736
       2018-02-28   180.721579   195.3200   167.1800   180.269474  516621991
       2018-03-31   173.449524   186.1000   149.0200   173.489524  996232472
       2018-04-30   164.163557   177.1000   150.5100   163.810476  751130388
       2018-05-31   181.910509   192.7200   170.2300   182.930000  401144183
       2018-06-30   194.974067   203.5500   186.4300   195.267619  387265765
       2018-07-31   199.332143   218.6200   166.5600   199.967143  652763259
       2018-08-31   177.598443   188.3000   170.2700   177.491957  549016789
       2018-09-30   164.232895   173.8900   158.8656   164.377368  500468912
       2018-10-31   154.873261   165.8800   139.0300   154.187826  622446235
       2018-11-30   141.762857   154.1300   126.8500   141.635714  518150415
       2018-12-31   137.529474   147.1900   123.0200   137.161053  558786249
GOOG   2018-01-31  1127.200952  1186.8900  1045.2300  1130.770476   28738485
       2018-02-28  1088.629474  1174.0000   992.5600  1088.206842   42384105
       2018-03-31  1096.108095  1177.0500   980.6400  1091.490476   45430049
       2018-04-30  1038.415238  1094.1600   990.3700  1035.696190   41773275
       2018-05-31  1064.021364  1110.7500  1006.2900  1069.275909   31849196
       2018-06-30  1136.396190  1186.2900  1096.0100  1137.626667   32103642
       2018-07-31  1183.464286  1273.8900  1093.8000  1187.590476   31953386
       2018-08-31  1226.156957  1256.5000  1188.2400  1225.671739   28820379
       2018-09-30  1176.878421  1212.9900  1146.9100  1175.808947   28863199
       2018-10-31  1116.082174  1209.9600   995.8300  1110.940435   48496167
       2018-11-30  1054.971429  1095.5700   996.0200  1056.162381   36735570
       2018-12-31  1042.620000  1124.6500   970.1100  1037.420526   40256461
NFLX   2018-01-31   231.269286   286.8100   195.4200   232.908095  238377533
       2018-02-28   270.873158   297.3600   236.1100   271.443684  184585819
       2018-03-31   31

In [32]:
#4
crosstab = pd.crosstab(
    earthquakes['tsunami'],
    earthquakes['magType'],
    values=earthquakes['mag'],
    aggfunc='max'
)

crosstab

magType,mb,mb_lg,md,mh,ml,ms_20,mw,mwb,mwr,mww
tsunami,,,,,,,,,,
0,5.6,3.5,4.11,1.1,4.2,NaN,3.83,5.8,4.8,6.0
1,6.1,NaN,NaN,NaN,5.1,5.7,4.41,NaN,NaN,7.5


In [33]:
#5
rolling_60 = (
    faang
    .groupby('ticker')
    .rolling(60)
    .agg({
        'open': 'mean',
        'high': 'max',
        'low': 'min',
        'close': 'mean',
        'volume': 'sum'
    })
)

rolling_60

open      high     low       close       volume
ticker date                                                             
AAPL   2018-01-02         NaN       NaN     NaN         NaN          NaN
       2018-01-03         NaN       NaN     NaN         NaN          NaN
       2018-01-04         NaN       NaN     NaN         NaN          NaN
       2018-01-05         NaN       NaN     NaN         NaN          NaN
       2018-01-08         NaN       NaN     NaN         NaN          NaN
...                       ...       ...     ...         ...          ...
NFLX   2018-12-24  306.018050  386.7999  233.68  303.239833  811001766.0
       2018-12-26  303.596050  386.7999  231.23  301.232167  818289623.0
       2018-12-27  301.500383  386.7999  231.23  299.134417  822148280.0
       2018-12-28  299.393050  380.9300  231.23  297.116750  824496849.0
       2018-12-31  297.420217  380.0000  231.23  295.293583  832207164.0

[1255 rows x 5 columns]

In [34]:
#6
pivot = pd.pivot_table(
    faang,
    index='ticker',
    values=['open', 'high', 'low', 'close', 'volume'],
    aggfunc='mean'
)

pivot

,close,high,low,open,volume
ticker,,,,,
AAPL,186.986218,188.906858,185.135729,187.038674,3.402145e+07
AMZN,1641.726175,1662.839801,1619.840398,1644.072669,5.649563e+06
FB,171.510936,173.615298,169.303110,171.454424,2.768798e+07
GOOG,1113.225139,1125.777649,1101.001594,1113.554104,1.742645e+06
NFLX,319.290299,325.224583,313.187273,319.620533,1.147030e+07


In [35]:
#7
nflx = faang[faang['ticker'] == 'NFLX'].copy()

numeric_cols = nflx.select_dtypes(include='number').columns

z_scores = nflx[numeric_cols].apply(
    lambda x: (x - x.mean()) / x.std()
)

z_scores

,open,high,low,close,volume
date,,,,,
2018-01-02,-2.500753,-2.516023,-2.410226,-2.416644,-0.088760
2018-01-03,-2.380291,-2.423180,-2.285793,-2.335286,-0.507606
2018-01-04,-2.296272,-2.406077,-2.234616,-2.323429,-0.959287
2018-01-05,-2.275014,-2.345607,-2.202087,-2.234303,-0.782331
2018-01-08,-2.218934,-2.295113,-2.143759,-2.192192,-1.038531
...,...,...,...,...,...
2018-12-24,-1.571478,-1.518366,-1.627197,-1.745946,-0.339003
2018-12-26,-1.735063,-1.439978,-1.677339,-1.341402,0.517040
2018-12-27,-1.407286,-1.417785,-1.495805,-1.302664,0.134868


In [36]:
#8
events = pd.DataFrame({
    'ticker': ['FB', 'FB', 'FB'],
    'date': pd.to_datetime(['2018-07-25', '2018-03-19', '2018-03-20']),
    'event': [
        'Disappointing user growth announced after close.',
        'Cambridge Analytica story',
        'FTC investigation'
    ]
})

events.set_index(['date', 'ticker'], inplace=True)

faang_multi = faang.reset_index().set_index(['date', 'ticker'])

merged = faang_multi.merge(
    events,
    how='outer',
    left_index=True,
    right_index=True
)

merged

open       high        low      close    volume event
date       ticker                                                            
2018-01-02 AAPL     166.9271   169.0264   166.0442   168.9872  25555934   NaN
           AMZN    1172.0000  1190.0000  1170.5100  1189.0100   2694494   NaN
           FB       177.6800   181.5800   177.5500   181.4200  18151903   NaN
           GOOG    1048.3400  1066.9400  1045.2300  1065.0000   1237564   NaN
           NFLX     196.1000   201.6500   195.4200   201.0700  10966889   NaN
...                      ...        ...        ...        ...       ...   ...
2018-12-31 AAPL     157.8529   158.6794   155.8117   157.0663  35003466   NaN
           AMZN    1510.8000  1520.7600  1487.0000  1501.9700   6954507   NaN
           FB       134.4500   134.6400   129.9500   131.0900  24625308   NaN
           GOOG    1050.9600  1052.7000  1023.5900  1035.6100   1493722   NaN
           NFLX     260.1600   270.1001   260.0000   267.6600  13508920   NaN

[1255 rows x 6 columns]

In [37]:
#9
indexed = (
    faang
    .groupby('ticker')
    .transform(lambda x: x / x.iloc[0])
)

indexed.head()

,open,high,low,close,volume
date,,,,,
2018-01-02,1.000000,1.000000,1.000000,1.000000,1.000000
2018-01-03,1.023638,1.017623,1.021290,1.017914,0.930292
2018-01-04,1.040635,1.025498,1.036889,1.016040,0.764707
2018-01-05,1.044518,1.029298,1.041566,1.029931,0.747830
2018-01-08,1.053579,1.040313,1.049451,1.037813,0.991341


## Conclusion
This hands-on activity was truly a challenge for me. It made use of the skills that were previously taught to us mixed with knowledge that I have yet to learn. In this activity, I had learned how to merge dataframes when I merged the faang.csv to the data that was given to me. I was also able to calculate complex matemathics, z-scores, on the dataframes. I got the aid of AI in understanding the errors that I encountered and what can I do to fix them. All in all, I truly enjoyed this activity and am eager to learn more in the following weeks to come.